In [ ]:
# =============================================================================
# ATAS Event Relabelling Notebook
#
# PURPOSE:
# This notebook updates automatically detected speech/pause segments by
# correcting mislabeled non-speech vocal events based on a reference CSV.
#
# INPUTS:
# - *_50_150.csv files (event-level segmentation output)
# - CSV file specifying indices of non-speech vocal events to relabel
#
# OUTPUTS:
# - *_50_150_changed.csv → row-level corrected labels (for verification)
# - *_50_150_final.csv   → collapsed segments (analysis-ready)
#
# WORKFLOW:
# 1. Load event-level CSV
# 2. Apply corrections to specified indices
# 3. Track modified rows
# 4. Collapse consecutive segments with the same label
#
# NOTE:
# The *_final.csv files are used in downstream analysis pipelines.
# =============================================================================

In [ ]:
import os, wave, itertools, statistics
import numpy as np, pandas as pd, librosa, librosa.display, matplotlib.pyplot as plt
import noisereduce as nr, IPython.display as ipd, matplotlib.patches as mpatches
import soundfile as sf
from scipy.io import wavfile
from scipy.stats import variation
from scipy.signal import butter, filtfilt
from pydub import AudioSegment, silence

In [ ]:
# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

In [ ]:
def automate_non_speech_vocal_events_update(
   csv_folder,
    changed_events_csv_filename
):

    # -------------------------------------------------------------------------
    # Load the CSV that contains filenames and indices of non-speech events
    # that need to be relabeled (corrections to apply)
    # -------------------------------------------------------------------------
    os.chdir(csv_folder)
    changed_events_csv = pd.read_csv(changed_events_csv_filename)

    # Column containing indices to update (comma-separated values)
    changed_events_ids = changed_events_csv['Non_speech_vocal_events']

    # Folder where individual event CSVs are stored
    folder = csv_folder

    # Dictionary to store updated DataFrames (for optional downstream use)
    updated_dfs = {}

    # -------------------------------------------------------------------------
    # Loop through each file listed in the correction CSV
    # -------------------------------------------------------------------------
    for _, row in changed_events_csv.iterrows():
        file_base = row['File_name']
        base_name = file_base.replace('.wav', '')

        # ---------------------------------------------------------------------
        # Define input and output file paths
        # ---------------------------------------------------------------------
        input_filename = f"{base_name}_50_150.csv"
        input_path = os.path.join(folder, input_filename)

        # Output with row-level corrections (no collapsing)
        changed_filename = f"{base_name}_50_150_changed.csv"
        changed_path = os.path.join(folder, changed_filename)

        # Final output with merged/collapsed segments
        final_filename = f"{base_name}_50_150_final.csv"
        final_path = os.path.join(folder, final_filename)

        try:
            # Load the original segmented event file
            df = pd.read_csv(input_path)

            # -----------------------------------------------------------------
            # Preserve original labels by renaming columns
            # -----------------------------------------------------------------
            if 'Labels' in df.columns:
                df = df.rename(columns={'Labels': 'Label_old'})
            if 'Labels_X' in df.columns:
                df = df.rename(columns={'Labels_X': 'Label_X_old'})

            # Create new working label column (copy of original)
            # and a column to track which rows are modified
            df['Labels'] = df['Label_old']
            df['changed_ids'] = 0

            # -----------------------------------------------------------------
            # Apply corrections:
            # Convert specified indices to 'p' (pause)
            # and mark them with changed_ids = 1
            # -----------------------------------------------------------------
            events_str = row.get('Non_speech_vocal_events', '')
            if pd.notna(events_str) and str(events_str).strip():
                try:
                    indices_to_update = [int(i.strip()) for i in str(events_str).split(',') if i.strip().isdigit()]
                    print(indices_to_update)

                    df.loc[df['index'].isin(indices_to_update), 'Labels'] = 'p'

                    matching_indices = df.loc[df['index'].isin(indices_to_update), 'index']
                    print(matching_indices.tolist())

                    df.loc[df['index'].isin(indices_to_update), 'changed_ids'] = 1

                except Exception as e:
                    print(f"Error parsing/applying Non_speech_vocal_events for {base_name}: {e}")

            # -----------------------------------------------------------------
            # Save row-level updated file (same structure as input)
            # This file preserves all segments but with corrected labels
            # -----------------------------------------------------------------
            updated_dfs[base_name] = df
            df.to_csv(changed_path, index=False)
            print(f"Saved updated file: {changed_path}")

            # -----------------------------------------------------------------
            # Collapse consecutive rows with the same label
            # This merges adjacent segments into longer continuous events
            # -----------------------------------------------------------------

            collapsed_rows = []
            start_idx = 0

            while start_idx < len(df):
                current_label = df.loc[start_idx, 'Labels']
                end_idx = start_idx

                # Extend the segment while the next row has the same label
                while end_idx + 1 < len(df) and df.loc[end_idx + 1, 'Labels'] == current_label:
                    end_idx += 1

                # Extract the group of consecutive rows
                group = df.iloc[start_idx:end_idx + 1]

                # -----------------------------------------------------------------
                # Create a single collapsed row:
                # - Start time from first row
                # - End time from last row
                # - Duration summed
                # - changed_ids = 1 if any row in the group was modified
                # -----------------------------------------------------------------
                collapsed_row = {
                    'level_0': group.iloc[0]['level_0'] if 'level_0' in group.columns else None,
                    'index': group.iloc[0]['index'],
                    'Time_start': group.iloc[0]['Time_start'],
                    'Time_end': group.iloc[-1]['Time_end'],
                    'Labels': current_label,
                    'Label_old': group.iloc[0]['Label_old'],
                    'Label_X_old': group.iloc[0]['Label_X_old'],
                    'Time_diff': group['Time_diff'].sum() if 'Time_diff' in group.columns else None,
                    'changed_ids': group['changed_ids'].max()
                }

                collapsed_rows.append(collapsed_row)
                start_idx = end_idx + 1

            # -----------------------------------------------------------------
            # Save collapsed (final) file
            # This is the cleaned, analysis-ready version with merged segments
            # -----------------------------------------------------------------
            final_df = pd.DataFrame(collapsed_rows)
            final_df.to_csv(final_path, index=False)
            print(f"Saved final collapsed file: {final_path}")

        except FileNotFoundError:
            # Handle missing input files gracefully
            print(f"File not found: {input_path}")

    return updated_dfs

In [ ]:
if __name__ == "__main__":
        automate_non_speech_vocal_events_update(csv_folder= base_dir +'/Individual_OutputCSV_Files',
        changed_events_csv_filename = base_dir + '/Stat_csv_files/CWNS_final_non_speech_vocal_events.csv'
    )

In [ ]:
if __name__ == "__main__":
        automate_non_speech_vocal_events_update(csv_folder= base_dir +'/Individual_OutputCSV_Files',
        changed_events_csv_filename = base_dir + '/Stat_csv_files/CWS_final_non_speech_vocal_events.csv'
    )